# Ensemble Methods (finally)

Ensemble methods combine predictions across multiple models to improve prediction.

 - *weak learners* - simple models that predict slightly better than random guessing
 - *strong learners* - more complex models that predict significantly better than chance


## Random Forests (RandomForestClassifier)

 A random forest creates numerous trees in parallel (at the same time). Each tree is relatively shallow, sometimes only a single node (called a stump).
 
 To make a prediction, a sample is processed by each tree, each tree makes a prediction, and the majority vote wins. But what makes the trees in the forest different from each other?

 In fitting, diversity of trees is created using two methods: feature selection and bagging.

### Random feature subsets

  Each tree only gets a subset of the features. For example, in a random forest deciding whether or not you should buy a car, one tree might make a prediction based on ['reliability', 'feul economy', 'price'], while another uses ['top speed', 'interior room', 'cost to repair'], and another uses ['resale value', 'feul economy', 'value of standard tech'] and another...

### Bagging (<ins>B</ins>ootstrap <ins>agg</ins>regat<ins>ing</ins>)

  *Bootstrapping* is a method for creating new data sets by sampling existing data sets. In bootstrapping, you select samples randomly and allow a sample to be selected multiple times (called sampling with replacement).

  *Bagging* is a method that uses bootstrapping to create different training sets for each tree, and then aggregating the results.

### Why it works?

  The idea is that no one tree will be great, but they'll all make different mistakes. But there'll be more overlap in correct guesses than in mistakes. So for any given sample, the majority vote is more likely to be correct than any one tree.

## Gradient Boosted Trees

Whereas Random Forests fit trees in parallel and every tree gets an equal vote, Boosted trees create trees sequentially, each new tree focusing on the shortcomings of the previous. And at the voting stage, some trees get more say than others.

There are many flavors of Boosted trees: AdaBoost, XGBoost, CatBoost

They all work a little differently, but here's an outline of AdaBoost as an example:

### AdaBoost (<ins>Ada</ins>ptive <ins>Boost</ins>ing) ( ```GradientBoostingClassifier```)

In AdaBoost, a tree comprises only one decision node; this kind of tree is called a stump. In each iteration, a new stump is created that splits the data based on a different condition. As the algorithm iterates, it keeps track of:

- Sample Weight - each iteration, the algorithm focuses more on misclassified samples. 
   - A sample that is classified correctly is down-weighted. We get this right, don't spend more energy on this case.
   - A sample that is classified incorrectly is up-weighted. We get this wrong, focus on this case.

 - Tree Influence - how much say a tree will have in the final vote. Trees that do better at classifying get more say.
    - A tree that is 50% correct gets no say. This tree is just guessing
    - A tree that is >50% gets a positive vote (0 to infinity). A tree that is 100% correct gets infinite vote! Listen to that tree!
    - A tree that is <50% gets a negative vote (0 to -infinity). A tree that is 0% correct gets a -infinite vote! Do the opposite of that tree!


The AdaBoost process:
 1. Start with all the samples each counts the same. 
 2. Same as in a decision tree, pick a question that splits the data to minimize Gini Impurity.
 3. Sum up sample weights for mis-classified samples and calculate Tree Influence.
 4. Assign new weights to samples, increasing weights on mistakes and decreasing weights on correct classifications.
 5. Create new stump, and repeat 2-5 until classification error is below some threshold you choose.

When you predict, you feed the sample through all the stumps and each votes according to their influence.

In [20]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
spambase = fetch_ucirepo(id=94) 
  
# data (as pandas dataframes) 
X = spambase.data.features 
Y = spambase.data.targets 

# metadata 
print(spambase.metadata) 
  
# variable information 
print(spambase.variables) 

{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split

# to make y a compatible shape for sklearn models
y = Y['Class']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Decision Tree
dt_params = {'max_depth': [3, 5, 10, None], 'min_samples_split': [2, 5, 10]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, n_jobs=-1)
dt_grid.fit(X_train, y_train)

# Random Forest
rf_params = {'n_estimators': [10, 30, 100], 'max_depth': [1, 2, 3]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)

# Gradient Boosted Trees
gb_params = {'n_estimators': [10, 30, 100], 'max_depth': [1, 2, 3]}
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42), gb_params, cv=5, n_jobs=-1)
gb_grid.fit(X_train, y_train)


# Get the best models
print("Best Decision Tree params:", dt_grid.best_params_)
print("Best Random Forest params:", rf_grid.best_params_)
print("Best Gradient Boosted params:", gb_grid.best_params_)

tree = dt_grid.best_estimator_
forest = rf_grid.best_estimator_
boosted = gb_grid.best_estimator_

Best Decision Tree params: {'max_depth': 10, 'min_samples_split': 5}
Best Random Forest params: {'max_depth': 3, 'n_estimators': 100}
Best Gradient Boosted params: {'max_depth': 3, 'n_estimators': 100}


In [ ]:
y_tree_train = tree.predict(X_train)
y_tree_test = tree.predict(X_test)

y_forest_train = forest.predict(X_train)
y_forest_test = forest.predict(X_test)

y_boosted_train = boosted.predict(X_train)
y_boosted_test = boosted.predict(X_test)


In [ ]:
from sklearn.metrics import confusion_
fig_tree, ax_tree = plt.subplots(2,1, figsize = (10,5))

